In [14]:
!pip install flask joblib
!pip install pyngrok
!ngrok authtoken 2rF3GEnzxoREZw1N2MSQji59AHA_4AUozkWuRyHibQwzGn7Bi
!pip install gunicorn
!pip install fuzzywuzzy
!pip install gunicorn
!pip install waitress
!pip install tensorflow_decision_forests
!pip install pandas scikit-learn fuzzywuzzy Flask joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from fuzzywuzzy import process
from sklearn.metrics import r2_score
import joblib
import numpy as np

In [17]:
# Load your dataset (appointments data with Appointment Type and Priority Score)
df = pd.read_excel("appointments_dataset.xlsx")

# Normalize the 'Appointment Type' column (lowercase and strip extra spaces)
df['Appointment Type'] = df['Appointment Type'].str.lower().str.strip()

# Encoding 'Appointment Type' column to numeric format using LabelEncoder
label_encoder = LabelEncoder()
df['Appointment Type Encoded'] = label_encoder.fit_transform(df['Appointment Type'])

# Features (Appointment Type Encoded) and Target (Priority Score)
X = df[['Appointment Type Encoded']]  # Input feature (encoded appointment type)
y = df['Priority']                   # Output (priority score)

# Split the data into training and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the model (RandomForestRegressor)
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

# Predict on the test set to check performance
y_pred = model.predict(X_test)

# Calculate the mean squared error for regression models
mse = mean_squared_error(y_test, y_pred)
print(f"Model Mean Squared Error: {mse:.2f}")
# Predict on the training data
y_train_pred = model.predict(X_train)

# Calculate the R-squared score for the training data

r2_train = r2_score(y_train, y_train_pred)

print(f"Model Training R-squared Score: {r2_train:.2f}")
# Predict on the test set
y_pred = model.predict(X_test)

# Calculate the R-squared score for the test data (testing accuracy)
r2 = r2_score(y_test, y_pred)

print(f"Model Testing R-squared Score: {r2:.2f}")
# Function to predict priority for a new appointment
def predict_priority_for_new_appointment(new_appointment):
    new_appointment = new_appointment.lower().strip()

    # Fuzzy matching to find the closest match
    closest_match = process.extractOne(new_appointment, df['Appointment Type'])

    if closest_match:
        matched_appointment = closest_match[0]
        new_appointment_encoded = label_encoder.transform([matched_appointment])

        # Ensure the input data has the same structure as the training data
        new_data = pd.DataFrame({'Appointment Type Encoded': new_appointment_encoded})

        # Predict priority
        predicted_priority = model.predict(new_data)
        return predicted_priority[0]
    else:
        return None
!fuser -k 5000/tcp


Model Mean Squared Error: 0.58
Model Training R-squared Score: 0.88
Model Testing R-squared Score: 0.84


In [22]:
# Example of predicting priority for a new appointment type
new_appointment = 'funeral'
predicted_priority = predict_priority_for_new_appointment(new_appointment)

if predicted_priority is not None:
    print(f"Predicted Priority Score for '{new_appointment}': {predicted_priority}")


# Save the model and the label encoder
joblib.dump(model, "priority_model.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")


# Save the label encoder using joblib
joblib.dump(label_encoder, 'appointment_label_encoder.pkl')
!pip install pyngrok  # Install pyngrok if not installed

# Authenticate ngrok
!ngrok authtoken 2rFePqdnKnIOAGn2mzlYWqgxawt_3RLm6a5zNehKepz3nj8rZ  # Replace with your authtoken

from flask import Flask, request, jsonify
import joblib
import numpy as np
from fuzzywuzzy import process

# Load the saved model and label encoder
model = joblib.load("priority_model.pkl")
label_encoder = joblib.load("label_encoder.pkl")

# Load the dataset for fuzzy matching
import pandas as pd
df = pd.read_excel("appointments_dataset.xlsx")
df['Appointment Type'] = df['Appointment Type'].str.lower().str.strip()

app = Flask(__name__)

@app.route('/predict_priority_v2', methods=['POST'])
def predict_priority_v2():
    try:
        # Get the appointment type from the request body
        data = request.get_json()
        appointment_type = data.get('appointment_type', '').lower().strip()

        # Predict the priority using your model
        predicted_priority = predict_priority_for_new_appointment(appointment_type)

        # Return the prediction as a JSON response
        return jsonify({'predicted_priority': predicted_priority})
    except Exception as e:
        return jsonify({'error': str(e)}), 400
from pyngrok import ngrok

# Open a public URL for the Flask app
public_url = ngrok.connect(5000)
print("Public URL:", public_url)


if __name__ == '__main__':
    app.run(debug=True)


Predicted Priority Score for 'funeral': 8.849999999999985


PyngrokNgrokError: The ngrok process was unable to start.